# Bella Goose Coffee — Cafe on the River: 3-Year Data Pipeline

**Author:** Zachari  
**Data Source:** Square POS Exports (Orders, Payments, Labor, Catalog)  
**Time Period:** March 2023 – March 2026  
**Timezone:** US/Mountain  

This notebook documents the complete pandas data pipeline used to extract, clean, and transform
Square POS data into the 30+ datasets powering the Bella Goose Business Intelligence Dashboard.

---

### Critical Data Caveat

> **The orders CSV has one row per ITEM, not per ORDER.**  
> Order-level fields (`order_total`, `order_discount`, `order_tip`, `order_tax`) are **duplicated**
> on every item row within a single order. If you naively sum these columns across all rows,
> you'll massively overcount. The fix: **deduplicate to order level** using
> `groupby('order_id').agg(..., 'first')` before summing order-level fields.  
> Item-level fields (`net_sales`, `discount`, `base_price`) are safe to sum across all rows.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Suppress pandas warnings for cleaner output
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Configuration
DATA_DIR = Path('.')  # Update to wherever your CSVs live
TIMEZONE = 'US/Mountain'
OUTPUT_FILE = 'dash_main_corrected.json'

# Season definitions (tourist town framing)
SEASON_MAP = {
    1: 'Off-Season', 2: 'Off-Season', 3: 'Off-Season', 4: 'Off-Season',
    5: 'Shoulder',    6: 'Peak',       7: 'Peak',       8: 'Peak',
    9: 'Shoulder',   10: 'Shoulder',  11: 'Off-Season', 12: 'Off-Season'
}

print("Config ready. Timezone:", TIMEZONE)

## 2. Load Raw Data

We load all four Square export CSVs. The orders file is the largest (~91MB, 728K rows)
because it contains one row per line item, not per order.

In [ ]:
# --- ORDERS (one row per ITEM — beware of order-level field duplication!) ---
orders = pd.read_csv(
    DATA_DIR / 'square_orders.csv',
    dtype={
        'order_id': str,
        'state': str,
        'source': str,
        'item_name': str,
        'modifiers': str
    },
    parse_dates=False  # We'll handle timezone conversion manually
)

print(f"Orders: {len(orders):,} rows, {orders['order_id'].nunique():,} unique orders")
print(f"Columns: {list(orders.columns)}")
orders.head(3)

In [ ]:
# --- PAYMENTS (one row per payment — no duplication issue) ---
payments = pd.read_csv(
    DATA_DIR / 'square_payments.csv',
    dtype={'payment_id': str, 'order_id': str, 'status': str, 'source_type': str, 'card_brand': str},
    parse_dates=False
)

print(f"Payments: {len(payments):,} rows")
payments.head(3)

In [ ]:
# --- LABOR (one row per shift) ---
labor = pd.read_csv(
    DATA_DIR / 'square_labor.csv',
    dtype={'shift_id': str, 'employee_id': str, 'status': str, 'job_title': str},
    parse_dates=False
)

print(f"Labor: {len(labor):,} rows, {labor['employee_id'].nunique()} unique employees")
labor.head(3)

In [ ]:
# --- CATALOG (menu items — used for reference only) ---
catalog = pd.read_csv(DATA_DIR / 'square_catalog.csv')
print(f"Catalog: {len(catalog):,} items")
catalog.head(3)

## 3. Timezone Conversion

Square exports timestamps in UTC (ISO 8601 format). Since the business operates in a Mountain
timezone tourist town, all analysis uses US/Mountain time. This matters for hourly heatmaps,
day-of-week analysis, and date groupings.

In [ ]:
# Convert orders timestamps to Mountain time
orders['created_at'] = (
    pd.to_datetime(orders['created_at'], format='ISO8601', utc=True)
    .dt.tz_convert(TIMEZONE)
)

# Extract time components for grouping
orders['year']  = orders['created_at'].dt.year
orders['month'] = orders['created_at'].dt.month
orders['ym']    = orders['created_at'].dt.to_period('M').astype(str)  # '2023-03' format
orders['dow']   = orders['created_at'].dt.day_name()
orders['hour']  = orders['created_at'].dt.hour
orders['quarter'] = orders['created_at'].dt.quarter
orders['season'] = orders['month'].map(SEASON_MAP)

print("Orders date range:", orders['created_at'].min().date(), "to", orders['created_at'].max().date())
print("Years covered:", sorted(orders['year'].unique()))
orders[['created_at', 'year', 'month', 'ym', 'dow', 'hour', 'season']].head()

In [ ]:
# Convert payments timestamps
payments['created_at'] = (
    pd.to_datetime(payments['created_at'], format='ISO8601', utc=True)
    .dt.tz_convert(TIMEZONE)
)
payments['year']  = payments['created_at'].dt.year
payments['month'] = payments['created_at'].dt.month
payments['ym']    = payments['created_at'].dt.to_period('M').astype(str)

print("Payments date range:", payments['created_at'].min().date(), "to", payments['created_at'].max().date())

In [ ]:
# Convert labor timestamps and compute shift hours
labor['start_at'] = (
    pd.to_datetime(labor['start_at'], format='ISO8601', utc=True)
    .dt.tz_convert(TIMEZONE)
)
labor['end_at'] = (
    pd.to_datetime(labor['end_at'], format='ISO8601', utc=True)
    .dt.tz_convert(TIMEZONE)
)
labor['hours'] = (labor['end_at'] - labor['start_at']).dt.total_seconds() / 3600
labor['cost']  = labor['hours'] * labor['hourly_rate']
labor['year']  = labor['start_at'].dt.year
labor['month'] = labor['start_at'].dt.month
labor['ym']    = labor['start_at'].dt.to_period('M').astype(str)

print(f"Labor date range: {labor['start_at'].min().date()} to {labor['start_at'].max().date()}")
print(f"Total labor hours: {labor['hours'].sum():,.0f}")
print(f"Total labor cost: ${labor['cost'].sum():,.0f}")

## 4. The Critical Fix: Order-Level Deduplication

**THIS IS THE MOST IMPORTANT STEP IN THE ENTIRE PIPELINE.**

The orders CSV has one row per *item*, not per *order*. Order-level fields like 
`order_total`, `order_discount`, and `order_tip` are duplicated on every item row 
within a single order.

For example, an order with 3 items has `order_discount = $3.50` on ALL THREE rows.
Naively summing would triple-count it.

**Wrong approach:** `orders['order_discount'].sum()` → overcounts by ~6.3x  
**Correct approach:** Deduplicate to one row per order first, THEN sum

This bug was caught by noticing $538K in discounts on $2.12M revenue (25%!) looked absurd.
The correct figure was $80.9K (3.7% — perfectly normal for a coffee shop with loyalty rewards).

In [ ]:
# --- Demonstrate the problem ---
wrong_discounts = orders['order_discount'].sum()
wrong_tips = orders['order_tip'].sum()
wrong_total = orders['order_total'].sum()

print("=== WRONG (naively summing all item rows) ===")
print(f"  Discounts: ${wrong_discounts:,.0f}")
print(f"  Tips:      ${wrong_tips:,.0f}")
print(f"  Total:     ${wrong_total:,.0f}")
print()

# --- The fix: deduplicate to order level ---
order_level = orders.groupby('order_id').agg(
    order_discount=('order_discount', 'first'),  # Take FIRST (they're all identical)
    order_tip=('order_tip', 'first'),
    order_total=('order_total', 'first'),
    order_tax=('order_tax', 'first'),
    state=('state', 'first'),
    created_at=('created_at', 'first'),
    year=('year', 'first'),
    month=('month', 'first'),
    ym=('ym', 'first'),
    quarter=('quarter', 'first'),
    season=('season', 'first'),
    dow=('dow', 'first'),
    hour=('hour', 'first'),
    # Item-level fields ARE safe to sum
    net_sales=('net_sales', 'sum'),
    item_discount=('discount', 'sum'),
    item_count=('quantity', 'sum')
).reset_index()

correct_discounts = order_level['order_discount'].sum()
correct_tips = order_level['order_tip'].sum()
correct_total = order_level['order_total'].sum()

print("=== CORRECT (deduplicated to order level) ===")
print(f"  Discounts: ${correct_discounts:,.0f}  (was overcounted {wrong_discounts/correct_discounts:.1f}x)")
print(f"  Tips:      ${correct_tips:,.0f}  (was overcounted {wrong_tips/correct_tips:.1f}x)")
print(f"  Total:     ${correct_total:,.0f}  (was overcounted {wrong_total/correct_total:.1f}x)")
print(f"\nOrder-level DataFrame: {len(order_level):,} rows (one per order)")

In [ ]:
# --- Cross-check tips against the payments table ---
# Payments table has no duplication issue, so it's our ground truth
payments_tips = payments[payments['status']=='COMPLETED']['tip'].sum()
deduped_tips  = order_level[order_level['state']=='COMPLETED']['order_tip'].sum()

print("=== TIP CROSS-CHECK ===")
print(f"  Payments table tips: ${payments_tips:,.0f}")
print(f"  Deduped orders tips: ${deduped_tips:,.0f}")
print(f"  Difference: ${abs(payments_tips - deduped_tips):,.0f}  ✓ Match!")

## 5. Filter to Completed Orders

We separate completed orders from cancelled/voided ones. Cancelled orders are tracked 
separately as a key metric (the cancellation rate spiked to 14% in 2025).

In [ ]:
# Split by completion status
completed_items  = orders[orders['state'] == 'COMPLETED'].copy()
completed_orders = order_level[order_level['state'] == 'COMPLETED'].copy()
cancelled_orders = order_level[order_level['state'] != 'COMPLETED'].copy()

print(f"Completed item rows: {len(completed_items):,}")
print(f"Completed orders:    {len(completed_orders):,}")
print(f"Cancelled orders:    {len(cancelled_orders):,}")
print(f"Overall cancel rate: {len(cancelled_orders)/len(order_level)*100:.1f}%")

## 6. Monthly Revenue & Order Trends

Group by year-month to create the primary time series. Revenue uses `net_sales` 
(item-level, safe to sum across all rows of completed items).

In [ ]:
# Monthly revenue from item-level net_sales (safe to sum directly)
monthly_rev = (
    completed_items
    .groupby('ym')['net_sales']
    .sum()
    .reset_index()
    .rename(columns={'net_sales': 'revenue'})
)

# Monthly order counts from deduplicated order-level data
monthly_orders = (
    completed_orders
    .groupby('ym')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'orders'})
)

monthly = monthly_rev.merge(monthly_orders, on='ym')
monthly = monthly.sort_values('ym')

# Build the output dataset
ds_monthly = {
    'labels':  monthly['ym'].tolist(),
    'revenue': monthly['revenue'].round(0).tolist(),
    'orders':  monthly['orders'].tolist()
}

print(f"Monthly data: {len(monthly)} months")
print(f"Revenue range: ${monthly['revenue'].min():,.0f} – ${monthly['revenue'].max():,.0f}")
monthly.head()

## 7. Year-over-Year Revenue Comparison

In [ ]:
# Monthly revenue by year — used for overlaid line chart
ds_yoy = {}
for yr in sorted(completed_items['year'].unique()):
    yr_data = completed_items[completed_items['year'] == yr]
    by_month = yr_data.groupby('month')['net_sales'].sum().reindex(range(1,13), fill_value=0)
    ds_yoy[str(yr)] = by_month.round(0).tolist()

# YoY growth rates
ds_yoy_growth = {
    'labels': ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
}
years = sorted(completed_items['year'].unique())
for i, yr in enumerate(years):
    if i == 0:
        continue  # No growth for first year
    prev_data = completed_items[completed_items['year'] == years[i-1]].groupby('month')['net_sales'].sum()
    curr_data = completed_items[completed_items['year'] == yr].groupby('month')['net_sales'].sum()
    growth = []
    for m in range(1, 13):
        p = prev_data.get(m, 0)
        c = curr_data.get(m, 0)
        growth.append(round(((c - p) / p * 100) if p > 0 else 0, 1))
    ds_yoy_growth[f'g{yr % 100}'] = growth

print("YoY data built for years:", list(ds_yoy.keys()))
print("2024 total:", f"${sum(ds_yoy.get('2024', [0])):,.0f}")
print("2025 total:", f"${sum(ds_yoy.get('2025', [0])):,.0f}")

## 8. Seasonal Revenue Analysis

Tourist town segmentation:
- **Peak**: June – August (summer tourist rush)
- **Shoulder**: May, September, October (transition months)
- **Off-Season**: November – April (locals-only)

In [ ]:
# Seasonal revenue breakdown by year
ds_seasonal = {}
for yr in sorted(completed_items['year'].unique()):
    yr_data = completed_items[completed_items['year'] == yr]
    by_season = yr_data.groupby('season')['net_sales'].sum()
    ds_seasonal[str(yr)] = {
        'peak':     round(by_season.get('Peak', 0)),
        'shoulder': round(by_season.get('Shoulder', 0)),
        'off':      round(by_season.get('Off-Season', 0))
    }

# Show the tourist-dependency story
for yr, s in ds_seasonal.items():
    total = sum(s.values())
    off_pct = s['off'] / total * 100 if total > 0 else 0
    print(f"{yr}: Peak ${s['peak']:,.0f} | Shoulder ${s['shoulder']:,.0f} | Off ${s['off']:,.0f} | Off-season share: {off_pct:.0f}%")
print("\nKey insight: Off-season share growing = less tourist-dependent")

## 9. Revenue Heatmap (Day of Week × Hour)

A 7×12 heatmap showing revenue by day-of-week and hour. This identifies the exact 
slots where the business makes the most money — critical for staffing.

In [ ]:
# Build pivot: rows = day of week, columns = hour, values = revenue
dow_order_full = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_order_abbr = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

heat_data = (
    completed_items
    .groupby(['dow', 'hour'])['net_sales']
    .sum()
    .reset_index()
)

# Filter to business hours (5am - 4pm)
heat_data = heat_data[(heat_data['hour'] >= 5) & (heat_data['hour'] <= 16)]

# Pivot — reindex with full day names (matching dt.day_name() output)
pivot = heat_data.pivot_table(index='dow', columns='hour', values='net_sales', fill_value=0)
pivot = pivot.reindex(dow_order_full)

# Format hour labels
hour_labels = [f"{h}:00" for h in pivot.columns]

ds_heatmap = {
    'rows': dow_order_abbr,  # Abbreviate for display
    'cols': hour_labels,
    'data': pivot.round(0).values.tolist()
}

print("Heatmap shape:", len(ds_heatmap['rows']), "x", len(ds_heatmap['cols']))
print("Peak cell value: $" + f"{pivot.values.max():,.0f}")

## 10. Day-of-Week Revenue & Traffic

In [ ]:
# Revenue and order count by day of week
dow_rev = (
    completed_items
    .groupby('dow')
    .agg(revenue=('net_sales', 'sum'))
    .reindex(dow_order_full)
)

dow_orders = (
    completed_orders
    .groupby('dow')['order_id']
    .count()
    .reindex(dow_order_full)
)

# Count unique dates to get average daily revenue
dow_dates = completed_items.groupby('dow')['created_at'].apply(lambda x: x.dt.date.nunique()).reindex(dow_order_full)
dow_avg = (dow_rev['revenue'] / dow_dates).round(0)

ds_dow = {
    'labels':  dow_order_abbr,  # Abbreviate for display
    'revenue': dow_rev['revenue'].round(0).tolist(),
    'avg_day': dow_avg.tolist(),
    'orders':  dow_orders.tolist()
}

for i, d in enumerate(dow_order_full):
    print(f"  {dow_order_abbr[i]}: ${dow_rev.loc[d, 'revenue']:,.0f} total, ${dow_avg[d]:,.0f}/day avg, {dow_orders[d]:,} orders")

## 11. Hourly Revenue Patterns by Season

In [ ]:
# Revenue by hour, split by season
ds_hourly = {'labels': [f"{h}:00" for h in range(5, 17)]}

for season_name, season_key in [('Peak', 'peak'), ('Shoulder', 'shoulder'), ('Off-Season', 'off')]:
    season_data = completed_items[completed_items['season'] == season_name]
    by_hour = (
        season_data
        .groupby('hour')['net_sales']
        .sum()
        .reindex(range(5, 17), fill_value=0)
    )
    ds_hourly[season_key] = by_hour.round(0).tolist()

print("Hourly data built for 3 seasons")
print("Peak hour 8am revenue:", f"${ds_hourly['peak'][3]:,.0f}")

## 12. Product Analysis — Top 25 Items

Item-level revenue and quantity by year. This uses `net_sales` (item-level) 
which is safe to sum directly from the raw orders CSV.

In [ ]:
# Top 25 items by total revenue across all years
item_yearly = (
    completed_items
    .groupby(['item_name', 'year'])
    .agg(revenue=('net_sales', 'sum'), qty=('quantity', 'sum'))
    .reset_index()
)

# Get overall top 25
top25_names = (
    completed_items
    .groupby('item_name')['net_sales']
    .sum()
    .nlargest(25)
    .index.tolist()
)

# Build item table
ds_item_table = []
for name in top25_names:
    item_data = item_yearly[item_yearly['item_name'] == name]
    row = {'name': name}
    total_rev = 0
    total_qty = 0
    for yr in sorted(completed_items['year'].unique()):
        yr_row = item_data[item_data['year'] == yr]
        rev = yr_row['revenue'].sum() if len(yr_row) > 0 else 0
        qty = yr_row['qty'].sum() if len(yr_row) > 0 else 0
        row[f'rev_{yr}'] = round(rev)
        row[f'qty_{yr}'] = int(qty)
        total_rev += rev
        total_qty += qty
    row['total_rev'] = round(total_rev)
    row['total_qty'] = int(total_qty)
    row['avg_price'] = round(total_rev / total_qty, 2) if total_qty > 0 else 0
    ds_item_table.append(row)

# Top 10 bar chart data
ds_top_items = {
    'labels': [r['name'] for r in ds_item_table[:10]],
    'values': [r['total_rev'] for r in ds_item_table[:10]]
}

print("Top 5 items by revenue:")
for r in ds_item_table[:5]:
    print(f"  {r['name']}: ${r['total_rev']:,} ({r['total_qty']:,} sold, avg ${r['avg_price']:.2f})")

## 13. Product Growth (YoY Item Revenue Change)

In [ ]:
# Calculate YoY growth for items with significant revenue
years = sorted(completed_items['year'].unique())
latest_yr = years[-1]
prev_yr = years[-2]

item_prev = (
    completed_items[completed_items['year'] == prev_yr]
    .groupby('item_name')['net_sales'].sum()
)
item_curr = (
    completed_items[completed_items['year'] == latest_yr]
    .groupby('item_name')['net_sales'].sum()
)

growth_df = pd.DataFrame({'prev': item_prev, 'curr': item_curr}).fillna(0)
growth_df['change'] = growth_df['curr'] - growth_df['prev']
growth_df['pct'] = ((growth_df['change'] / growth_df['prev']) * 100).replace([np.inf, -np.inf], 0)

# Top 10 by absolute change (mix of gainers and losers)
top_growth = growth_df.nlargest(5, 'change')
top_decline = growth_df.nsmallest(5, 'change')
product_growth_items = pd.concat([top_growth, top_decline]).sort_values('change', ascending=False)

ds_product_growth = {
    'labels': product_growth_items.index.tolist(),
    'values': product_growth_items['change'].round(0).tolist(),
    'pct':    product_growth_items['pct'].round(1).tolist()
}

print("Biggest gainers:")
for name, row in top_growth.iterrows():
    print(f"  {name}: +${row['change']:,.0f} ({row['pct']:+.1f}%)")
print("\nBiggest decliners:")
for name, row in top_decline.iterrows():
    print(f"  {name}: ${row['change']:,.0f} ({row['pct']:+.1f}%)")

## 14. Drink vs Food Split & Attach Rate

Classifies items as Drink or Food to track category mix over time. 
The "attach rate" measures what % of orders include at least one food item — 
a key metric for average order value.

In [ ]:
# Simple classification (customize based on your menu)
food_keywords = ['sammie', 'waffle', 'toast', 'cookie', 'brownie', 'bread', 'muffin',
                 'burrito', 'bagel', 'cake', 'pie', 'scone', 'croissant', 'sandwich',
                 'egg', 'bowl', 'wrap', 'salad', 'quiche', 'roll', 'bar', 'biscuit']

def classify_item(name):
    if pd.isna(name):
        return 'Other'
    name_lower = name.lower()
    return 'Food' if any(kw in name_lower for kw in food_keywords) else 'Drink'

completed_items['category'] = completed_items['item_name'].apply(classify_item)

# Yearly drink/food split
ds_drink_food = {}
for yr in sorted(completed_items['year'].unique()):
    yr_data = completed_items[completed_items['year'] == yr]
    by_cat = yr_data.groupby('category')['net_sales'].sum()
    ds_drink_food[str(yr)] = {
        'drink': round(by_cat.get('Drink', 0)),
        'food':  round(by_cat.get('Food', 0))
    }

# Attach rate: % of orders with at least one food item
ds_attach_rate = {}
for yr in sorted(completed_items['year'].unique()):
    yr_items = completed_items[completed_items['year'] == yr]
    yr_orders = yr_items['order_id'].nunique()
    food_orders = yr_items[yr_items['category'] == 'Food']['order_id'].nunique()
    ds_attach_rate[str(yr)] = round(food_orders / yr_orders * 100, 1) if yr_orders > 0 else 0

print("Drink/Food split by year:")
for yr, split in ds_drink_food.items():
    total = split['drink'] + split['food']
    print(f"  {yr}: Drink ${split['drink']:,} ({split['drink']/total*100:.0f}%) | Food ${split['food']:,} ({split['food']/total*100:.0f}%)")

print("\nAttach rate trend:", ds_attach_rate)
print("Key insight: Declining attach rate correlates with Egg Sammie discontinuation")

## 15. Order Composition (Drink Only / Food Only / Combo)

In [ ]:
# Classify each order by what it contains
order_categories = (
    completed_items
    .groupby('order_id')['category']
    .apply(set)
    .reset_index()
)
order_categories['type'] = order_categories['category'].apply(
    lambda cats: 'Combo' if ('Drink' in cats and 'Food' in cats)
    else ('Food Only' if 'Food' in cats else 'Drink Only')
)

# Merge with order-level AOV
order_aov = completed_orders[['order_id', 'net_sales']].copy()
order_merged = order_categories.merge(order_aov, on='order_id')

comp_stats = order_merged.groupby('type').agg(
    count=('order_id', 'count'),
    aov=('net_sales', 'mean')
)

type_order = ['Drink Only', 'Food Only', 'Combo']
comp_stats = comp_stats.reindex(type_order)

ds_order_comp = {
    'labels': type_order,
    'counts': comp_stats['count'].tolist(),
    'aov':    comp_stats['aov'].round(2).tolist()
}

for t in type_order:
    print(f"  {t}: {comp_stats.loc[t, 'count']:,} orders, AOV ${comp_stats.loc[t, 'aov']:.2f}")
print(f"\nCombo AOV is {comp_stats.loc['Combo','aov']/comp_stats.loc['Drink Only','aov']:.1f}x higher than Drink Only")

## 16. Popular Modifiers (Add-ons)

In [ ]:
# Parse modifiers — they're stored as a single string field
mods = completed_items[completed_items['modifiers'].notna()]['modifiers']

# Split and count individual modifiers
mod_counts = {}
for mod_str in mods:
    for mod in str(mod_str).split(','):
        mod = mod.strip()
        if mod and mod != 'nan':
            mod_counts[mod] = mod_counts.get(mod, 0) + 1

# Top 15 modifiers
top_mods = sorted(mod_counts.items(), key=lambda x: -x[1])[:15]

ds_modifiers = {
    'labels': [m[0] for m in top_mods],
    'values': [m[1] for m in top_mods]
}

print("Top 10 modifiers:")
for name, count in top_mods[:10]:
    print(f"  {name}: {count:,}")

## 17. Average Order Value (AOV) Trend

In [ ]:
# Monthly AOV from deduplicated order-level data
monthly_aov = (
    completed_orders
    .groupby('ym')
    .agg(aov=('net_sales', 'mean'))
    .reset_index()
    .sort_values('ym')
)

ds_aov = {
    'labels': monthly_aov['ym'].tolist(),
    'values': monthly_aov['aov'].round(2).tolist()
}

print(f"AOV range: ${monthly_aov['aov'].min():.2f} – ${monthly_aov['aov'].max():.2f}")
print(f"Overall AOV: ${completed_orders['net_sales'].mean():.2f}")

## 18. Labor Analysis

Shift-level data showing labor costs, hours, and efficiency metrics. 
Critical for understanding the biggest controllable expense.

In [ ]:
# Monthly labor cost and hours
monthly_labor = (
    labor
    .groupby('ym')
    .agg(cost=('cost', 'sum'), hours=('hours', 'sum'))
    .reset_index()
    .sort_values('ym')
)

ds_labor_monthly = {
    'labels': monthly_labor['ym'].tolist(),
    'values': monthly_labor['cost'].round(0).tolist(),
    'hours':  monthly_labor['hours'].round(0).tolist()
}

print(f"Monthly labor data: {len(monthly_labor)} months")
print(f"Total labor: ${monthly_labor['cost'].sum():,.0f}, {monthly_labor['hours'].sum():,.0f} hours")

In [ ]:
# Revenue vs Labor overlay (merge on ym)
rev_labor = monthly_rev.merge(monthly_labor[['ym', 'cost']], on='ym', how='outer').sort_values('ym').fillna(0)

ds_rev_vs_labor = {
    'labels':  rev_labor['ym'].tolist(),
    'revenue': rev_labor['revenue'].round(0).tolist(),
    'labor':   rev_labor['cost'].round(0).tolist()
}

# Labor as % of revenue
rev_labor['pct'] = (rev_labor['cost'] / rev_labor['revenue'] * 100).round(1)

ds_labor_pct = {
    'labels': rev_labor['ym'].tolist(),
    'values': rev_labor['pct'].tolist()
}

print("Labor % of revenue:")
print(f"  Average: {rev_labor['pct'].mean():.1f}%")
print(f"  Range: {rev_labor['pct'].min():.1f}% – {rev_labor['pct'].max():.1f}%")

In [ ]:
# Labor by role
role_stats = (
    labor
    .groupby('job_title')
    .agg(
        cost=('cost', 'sum'),
        hours=('hours', 'sum'),
        employees=('employee_id', 'nunique')
    )
    .sort_values('cost', ascending=False)
    .reset_index()
)

ds_labor_roles = role_stats.rename(columns={'job_title': 'title'}).to_dict('records')
for r in ds_labor_roles:
    r['cost'] = round(r['cost'])
    r['hours'] = round(r['hours'])

print("Labor by role:")
for r in ds_labor_roles[:6]:
    print(f"  {r['title']}: ${r['cost']:,} ({r['hours']:,} hrs, {r['employees']} people)")

In [ ]:
# Revenue per labor hour (efficiency metric)
rev_hours = monthly_rev.merge(monthly_labor[['ym', 'hours']], on='ym', how='inner').sort_values('ym')
rev_hours['rev_per_hour'] = (rev_hours['revenue'] / rev_hours['hours']).round(2)

ds_rev_per_hour = {
    'labels': rev_hours['ym'].tolist(),
    'values': rev_hours['rev_per_hour'].tolist()
}

print(f"Revenue per labor hour: ${rev_hours['rev_per_hour'].mean():.2f} avg")
print(f"  Range: ${rev_hours['rev_per_hour'].min():.2f} – ${rev_hours['rev_per_hour'].max():.2f}")

In [ ]:
# Employee retention (unique employees per year)
ds_retention = {}
for yr in sorted(labor['year'].unique()):
    yr_labor = labor[labor['year'] == yr]
    ds_retention[str(yr)] = int(yr_labor['employee_id'].nunique())

print("Unique employees per year:", ds_retention)

In [ ]:
# Average wage trend
wage_monthly = (
    labor
    .groupby('ym')['hourly_rate']
    .mean()
    .reset_index()
    .sort_values('ym')
)

ds_wage_trend = {
    'labels': wage_monthly['ym'].tolist(),
    'values': wage_monthly['hourly_rate'].round(2).tolist()
}

print(f"Avg hourly rate: ${wage_monthly['hourly_rate'].mean():.2f}")
print(f"  Start: ${wage_monthly['hourly_rate'].iloc[0]:.2f} → End: ${wage_monthly['hourly_rate'].iloc[-1]:.2f}")

## 19. Financial Metrics

Discounts, tips, processing fees, payment methods, gift cards, and cancellations. 
**All order-level metrics use the deduplicated `order_level` DataFrame.**

In [ ]:
# --- DISCOUNTS (quarterly, from deduplicated order data) ---
# Compute gross sales per quarter for the rate
quarterly_gross = (
    completed_items
    .assign(yq=lambda x: x['year'].astype(str) + ' Q' + x['quarter'].astype(str))
    .groupby('yq')
    .agg(gross=('gross_sales', 'sum'))
)

quarterly_disc = (
    completed_orders
    .assign(yq=lambda x: x['year'].astype(str) + ' Q' + x['quarter'].astype(str))
    .groupby('yq')
    .agg(discount=('order_discount', 'sum'))
)

disc_merged = quarterly_gross.join(quarterly_disc).fillna(0)
disc_merged['rate'] = (disc_merged['discount'] / disc_merged['gross'] * 100).round(1)
disc_merged = disc_merged.sort_index()

ds_discounts = {
    'labels': disc_merged.index.tolist(),
    'rate':   disc_merged['rate'].tolist(),
    'total':  disc_merged['discount'].round(0).tolist()
}

print("Discount rate by quarter:")
for idx, row in disc_merged.iterrows():
    print(f"  {idx}: {row['rate']:.1f}% (${row['discount']:,.0f})")

In [ ]:
# --- TIPS (monthly, from deduplicated order data) ---
monthly_tips = (
    completed_orders
    .groupby('ym')
    .agg(
        total_tips=('order_tip', 'sum'),
        order_count=('order_id', 'count'),
        total_sales=('net_sales', 'sum')
    )
    .reset_index()
    .sort_values('ym')
)
monthly_tips['avg_tip'] = (monthly_tips['total_tips'] / monthly_tips['order_count']).round(2)
monthly_tips['tip_rate'] = (monthly_tips['total_tips'] / monthly_tips['total_sales'] * 100).round(1)

ds_tips = {
    'labels': monthly_tips['ym'].tolist(),
    'total':  monthly_tips['total_tips'].round(0).tolist(),
    'avg':    monthly_tips['avg_tip'].tolist(),
    'rate':   monthly_tips['tip_rate'].tolist()
}

print(f"Total tips (3-year): ${monthly_tips['total_tips'].sum():,.0f}")
print(f"Avg tip per order: ${monthly_tips['avg_tip'].mean():.2f}")
print(f"Avg tip rate: {monthly_tips['tip_rate'].mean():.1f}%")

In [ ]:
# --- PROCESSING FEES (from payments table) ---
completed_pmts = payments[payments['status'] == 'COMPLETED'].copy()

monthly_fees = (
    completed_pmts
    .groupby('ym')
    .agg(fees=('processing_fees', 'sum'), amount=('amount', 'sum'))
    .reset_index()
    .sort_values('ym')
)
monthly_fees['rate'] = (monthly_fees['fees'] / monthly_fees['amount'] * 100).round(2)

ds_fees = {
    'labels': monthly_fees['ym'].tolist(),
    'amount': monthly_fees['fees'].round(0).tolist(),
    'rate':   monthly_fees['rate'].tolist()
}

print(f"Total processing fees: ${monthly_fees['fees'].sum():,.0f}")
print(f"Avg fee rate: {monthly_fees['rate'].mean():.2f}%")

In [ ]:
# --- PAYMENT METHODS (by year) ---
ds_payment_trend = {}
for yr in sorted(completed_pmts['year'].unique()):
    yr_data = completed_pmts[completed_pmts['year'] == yr]
    by_method = yr_data.groupby('source_type')['amount'].sum().sort_values(ascending=False)
    ds_payment_trend[str(yr)] = {
        method: round(amt) for method, amt in by_method.items()
    }

print("Payment methods by year:")
for yr, methods in ds_payment_trend.items():
    print(f"  {yr}: {methods}")

In [ ]:
# --- GIFT CARDS (from payments where source_type contains 'GIFT' or card_brand is gift) ---
# Look for gift card indicators in the payments
gift_mask = (
    completed_pmts['source_type'].str.contains('SQUARE_GIFT_CARD', case=False, na=False) |
    completed_pmts['card_brand'].str.contains('SQUARE_GIFT_CARD', case=False, na=False)
)

gift_monthly = (
    completed_pmts[gift_mask]
    .groupby('ym')
    .agg(amount=('amount', 'sum'), count=('payment_id', 'count'))
    .reindex(monthly_fees['ym'], fill_value=0)
    .reset_index()
)

ds_gift_cards = {
    'labels': gift_monthly['ym'].tolist(),
    'amount': gift_monthly['amount'].round(0).tolist() if gift_monthly['amount'].sum() > 0 else [0] * len(gift_monthly),
    'count':  gift_monthly['count'].tolist() if gift_monthly['count'].sum() > 0 else [0] * len(gift_monthly)
}

print(f"Gift card transactions: {gift_monthly['count'].sum():,}")
print(f"Gift card revenue: ${gift_monthly['amount'].sum():,.0f}")

In [ ]:
# --- CANCELLATIONS (by year, from order-level data) ---
ds_cancellations = {}
for yr in sorted(order_level['year'].unique()):
    yr_data = order_level[order_level['year'] == yr]
    total = len(yr_data)
    cancelled = len(yr_data[yr_data['state'] != 'COMPLETED'])
    rate = round(cancelled / total * 100, 1) if total > 0 else 0
    ds_cancellations[str(yr)] = {
        'total': int(total),
        'cancelled': int(cancelled),
        'rate': rate
    }

print("Cancellation rates by year:")
for yr, stats in ds_cancellations.items():
    print(f"  {yr}: {stats['cancelled']:,} / {stats['total']:,} = {stats['rate']}%")
print("\n⚠️  14% cancellation rate in 2025 is a major red flag — potential $170K revenue leak")

## 20. Profit & Loss Summary

Combines revenue, discounts, labor, processing fees, and tips into a yearly P&L. 
All order-level metrics use deduplicated data.

In [ ]:
# Build P&L by year
ds_pnl = {}
for yr in sorted(completed_items['year'].unique()):
    # Gross revenue (item-level, safe to sum)
    gross = completed_items[completed_items['year'] == yr]['gross_sales'].sum()
    
    # Discounts (order-level, MUST use deduplicated data)
    discounts = completed_orders[completed_orders['year'] == yr]['order_discount'].sum()
    
    # Net revenue
    net_revenue = gross - discounts
    
    # Labor cost
    labor_cost = labor[labor['year'] == yr]['cost'].sum()
    
    # Processing fees (from payments)
    yr_fees = completed_pmts[completed_pmts['year'] == yr]['processing_fees'].sum()
    
    # Tips (order-level, deduplicated)
    tips = completed_orders[completed_orders['year'] == yr]['order_tip'].sum()
    
    # Margin after labor
    margin = net_revenue - labor_cost
    margin_pct = round(margin / net_revenue * 100, 1) if net_revenue > 0 else 0
    
    ds_pnl[str(yr)] = {
        'gross':               round(gross),
        'discounts':           round(discounts),
        'net_revenue':         round(net_revenue),
        'labor':               round(labor_cost),
        'fees':                round(yr_fees),
        'tips':                round(tips),
        'margin_after_labor':  round(margin),
        'margin_pct':          margin_pct
    }

print("P&L Summary:")
print(f"{'':>6} {'Gross':>12} {'Disc':>10} {'Net Rev':>12} {'Labor':>10} {'Fees':>8} {'Tips':>10} {'Margin':>12} {'M%':>6}")
for yr, p in ds_pnl.items():
    print(f"{yr:>6} ${p['gross']:>10,} ${p['discounts']:>8,} ${p['net_revenue']:>10,} ${p['labor']:>8,} ${p['fees']:>6,} ${p['tips']:>8,} ${p['margin_after_labor']:>10,} {p['margin_pct']:>5.1f}%")

## 21. Export to JSON

All 29 datasets are packaged into a single JSON file. This file is consumed by
the `build_dashboard.py` script to construct the self-contained HTML dashboard.

In [ ]:
# Assemble all datasets
output = {
    'monthly':        ds_monthly,
    'yoy':            ds_yoy,
    'yoy_growth':     ds_yoy_growth,
    'seasonal':       ds_seasonal,
    'heatmap':        ds_heatmap,
    'dow':            ds_dow,
    'hourly':         ds_hourly,
    'item_table':     ds_item_table,
    'top_items':      ds_top_items,
    'product_growth': ds_product_growth,
    'drink_food_yearly': ds_drink_food,
    'order_comp':     ds_order_comp,
    'attach_rate':    ds_attach_rate,
    'modifiers':      ds_modifiers,
    'aov':            ds_aov,
    'labor_monthly':  ds_labor_monthly,
    'rev_vs_labor':   ds_rev_vs_labor,
    'labor_pct':      ds_labor_pct,
    'labor_roles':    ds_labor_roles,
    'rev_per_hour':   ds_rev_per_hour,
    'retention':      ds_retention,
    'wage_trend':     ds_wage_trend,
    'discounts':      ds_discounts,
    'tips':           ds_tips,
    'fees':           ds_fees,
    'payment_trend':  ds_payment_trend,
    'gift_cards':     ds_gift_cards,
    'cancellations':  ds_cancellations,
    'pnl':            ds_pnl
}

# Write JSON
with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\nExported {len(output)} datasets to {OUTPUT_FILE}")
print(f"File size: {Path(OUTPUT_FILE).stat().st_size / 1024:.1f} KB")
print("\nDatasets:", list(output.keys()))

## Summary

This notebook documents the complete data pipeline for the Bella Goose Coffee 3-year analysis:

1. **Data Loading** — 4 Square POS CSV exports (orders, payments, labor, catalog)
2. **Timezone Conversion** — UTC → US/Mountain for all timestamps
3. **Order-Level Deduplication** — The critical fix for order_discount/tip/total overcounting
4. **29 Analytical Datasets** — Revenue trends, YoY, seasonal, heatmap, products, labor, financials, P&L
5. **JSON Export** — Single file consumed by the HTML dashboard builder

### Key Lessons Learned

- **Always inspect your grain.** The orders CSV being item-level (not order-level) caused a 6.3x overcount on discounts until caught.
- **Cross-check with independent sources.** The payments table confirmed the corrected tip totals matched within $7 across 291K+ orders.
- **Tourist town framing matters.** Summer peaks aren't anomalies — they're expected. The real story is how off-season performance is growing.
